# PRR Hypothesis Testing — Pediatric vs Adult Signal Profiles

## Main Hypothesis
- **H0-2:** Pediatric and adult PRR-filtered signal profiles are NOT meaningfully different
- **H1-2:** Pediatric and adult PRR-filtered signal profiles differ substantially, so pooled analysis may mask cohort-specific patterns

## Sub-Hypotheses
### H1-2A: Signal set overlap
- **H0-2A:** Pediatric and adult PRR-signal sets overlap sufficiently to justify pooled analysis
- **H1-2A:** Pediatric and adult PRR-signal sets have LOW overlap → distinct signal repertoires

### H1-2B: Shared signal strength
- **H0-2B:** For shared drug-AE pairs, signal strength does NOT differ significantly between cohorts
- **H1-2B:** Shared signals differ significantly in magnitude between pediatric and adult

### H1-2C: Pooled vs stratified
- **H0-2C:** Pooled PRR analysis does NOT miss cohort-specific signals significantly
- **H1-2C:** Pooled PRR analysis misses cohort-specific signals → stratification is necessary

**Data source:** PRR-filtered datasets from `signal_filtering_and_stratification.ipynb`
**Output:** `data/notebook/output/hypothesis_prr/`

In [ ]:
# ============================================================
# Setup + Load PRR-filtered datasets
# ============================================================
from pathlib import Path
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import norm, pearsonr, spearmanr
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

# FDR correction — fallback if statsmodels unavailable
try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("statsmodels not found — using manual BH FDR implementation")

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

ROOT = Path("..").resolve()
SIGNAL_OUT = ROOT / "data" / "notebook" / "output" / "signal_analysis"
OUTPUT_DIR = ROOT / "data" / "notebook" / "output" / "hypothesis_prr"
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# Load PRR-filtered datasets
adult_prr = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "adult_prr_filtered.parquet")
ped_prr = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "pediatric_prr_filtered.parquet")
pooled_prr = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "adult_and_ped_prr_filtered.parquet")

print(f"Adult PRR-filtered:    {adult_prr.height:,} pairs")
print(f"Pediatric PRR-filtered:{ped_prr.height:,} pairs")
print(f"Pooled PRR-filtered:   {pooled_prr.height:,} pairs")

In [ ]:
# ============================================================
# Helper functions
# ============================================================
def pair_key_set(df):
    """Return set of (drug, event) tuples from a polars DataFrame."""
    return set(zip(df["drug"].to_list(), df["event"].to_list()))

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0

def overlap_coefficient(a, b):
    """|A ∩ B| / min(|A|, |B|)"""
    denom = min(len(a), len(b))
    return len(a & b) / denom if denom > 0 else 0.0

def bh_fdr(pvals):
    """Benjamini-Hochberg FDR adjustment (manual implementation)."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    # Enforce monotonicity
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = np.empty(n)
    out[order] = q
    return out

print("Helpers defined: pair_key_set, jaccard, overlap_coefficient, bh_fdr")

---
# Section 1 — PRR Overlap Between Pediatric and Adult Signal Sets

Compute signal set overlap metrics to test **H1-2A**.

In [ ]:
# Build sets
adult_set = pair_key_set(adult_prr)
ped_set   = pair_key_set(ped_prr)
pooled_set = pair_key_set(pooled_prr)

shared = adult_set & ped_set
adult_only = adult_set - ped_set
ped_only = ped_set - adult_set

n_adult, n_ped, n_pooled = len(adult_set), len(ped_set), len(pooled_set)
n_shared = len(shared)
n_adult_only = len(adult_only)
n_ped_only = len(ped_only)

jac = jaccard(adult_set, ped_set)
ovc = overlap_coefficient(adult_set, ped_set)
shared_over_adult = n_shared / n_adult if n_adult else 0
shared_over_ped = n_shared / n_ped if n_ped else 0

# Build Table PRR-Overlap
overlap_table = pd.DataFrame([
    {"Metric": "Adult PRR-positive pairs",        "Value": f"{n_adult:,}"},
    {"Metric": "Pediatric PRR-positive pairs",    "Value": f"{n_ped:,}"},
    {"Metric": "Pooled PRR-positive pairs",       "Value": f"{n_pooled:,}"},
    {"Metric": "Shared pairs (Adult ∩ Pediatric)", "Value": f"{n_shared:,}"},
    {"Metric": "Adult-only pairs",                 "Value": f"{n_adult_only:,}"},
    {"Metric": "Pediatric-only pairs",             "Value": f"{n_ped_only:,}"},
    {"Metric": "Jaccard index",                    "Value": f"{jac:.4f}"},
    {"Metric": "Overlap coefficient",              "Value": f"{ovc:.4f}"},
    {"Metric": "Shared / Adult",                   "Value": f"{shared_over_adult:.4f}"},
    {"Metric": "Shared / Pediatric",               "Value": f"{shared_over_ped:.4f}"},
])
display(Markdown("**Table PRR-Overlap:**"))
display(overlap_table)
overlap_table.to_csv(OUTPUT_DIR / "tables" / "table_prr_overlap.csv", index=False)

# Bar plot
fig, ax = plt.subplots(figsize=(10, 5))
cats = ["Adult-only", "Shared", "Pediatric-only"]
vals = [n_adult_only, n_shared, n_ped_only]
colors = ["#2196F3", "#4CAF50", "#FF9800"]
bars = ax.bar(cats, vals, color=colors, edgecolor="white", alpha=0.9, width=0.6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
            f"{v:,}", ha="center", fontweight="bold")
ax.set_ylabel("Number of Drug-AE Pairs")
ax.set_title(f"PRR Signal Overlap — Jaccard = {jac:.3f}, Overlap Coef = {ovc:.3f}")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "S1_prr_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

### Section 1 — Interpretation

**Key thresholds for H1-2A support:**
- Jaccard < 0.30 → LOW overlap → **supports H1-2A**
- Jaccard 0.30–0.60 → moderate overlap
- Jaccard > 0.60 → high overlap → fails to support H1-2A

**Observation:** With low Jaccard and most pairs being cohort-specific, pediatric and adult
PRR-signal sets have distinct signal repertoires. This **supports H1-2A** — analyzing them
pooled would obscure the cohort-specific profile.

---
# Section 2 — Shared PRR Pairs: Magnitude Comparison + Heterogeneity Testing

For pairs in `shared` (both cohorts passed PRR), compare log(PRR) magnitudes.

**SE(log(PRR))** = √(1/a − 1/(a+b) + 1/c − 1/(c+d))  
**Z** = (log(PRR_ped) − log(PRR_adult)) / √(SE_ped² + SE_adult²)  
**FDR** = Benjamini-Hochberg adjusted p-values

Tests **H1-2B**.

In [ ]:
# Build shared dataframe — join adult_prr + ped_prr on (drug, event)
shared_df = pd.DataFrame({"drug": [p[0] for p in shared], "event": [p[1] for p in shared]})

adult_pd = adult_prr.select(
    "drug", "event", "a", "b", "c", "d",
    "PRR", "PRR_lower", "PRR_upper", "ROR", "EBGM"
).to_pandas().rename(columns={
    "a": "a_adult", "b": "b_adult", "c": "c_adult", "d": "d_adult",
    "PRR": "PRR_adult", "PRR_lower": "PRR_lower_adult", "PRR_upper": "PRR_upper_adult",
    "ROR": "ROR_adult", "EBGM": "EBGM_adult"
})
ped_pd = ped_prr.select(
    "drug", "event", "a", "b", "c", "d",
    "PRR", "PRR_lower", "PRR_upper", "ROR", "EBGM"
).to_pandas().rename(columns={
    "a": "a_ped", "b": "b_ped", "c": "c_ped", "d": "d_ped",
    "PRR": "PRR_ped", "PRR_lower": "PRR_lower_ped", "PRR_upper": "PRR_upper_ped",
    "ROR": "ROR_ped", "EBGM": "EBGM_ped"
})

merged = shared_df.merge(adult_pd, on=["drug", "event"], how="inner").merge(ped_pd, on=["drug", "event"], how="inner")
print(f"Shared pairs: {len(merged):,}")

# Compute log(PRR) + SE per cohort
def se_logprr(a, b, c, d):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))

merged["logPRR_adult"] = np.log(merged["PRR_adult"])
merged["logPRR_ped"]   = np.log(merged["PRR_ped"])
merged["log_ratio"]    = merged["logPRR_ped"] - merged["logPRR_adult"]

merged["SE_adult"] = se_logprr(merged["a_adult"], merged["b_adult"], merged["c_adult"], merged["d_adult"])
merged["SE_ped"]   = se_logprr(merged["a_ped"], merged["b_ped"], merged["c_ped"], merged["d_ped"])

# z-test
merged["Z"] = merged["log_ratio"] / np.sqrt(merged["SE_ped"]**2 + merged["SE_adult"]**2)
merged["p_raw"] = 2 * (1 - norm.cdf(np.abs(merged["Z"])))

# Keep only finite/valid rows for FDR
valid_mask = merged["p_raw"].notna() & np.isfinite(merged["p_raw"])
merged["p_fdr"] = np.nan
if HAS_STATSMODELS:
    rej, padj, _, _ = multipletests(merged.loc[valid_mask, "p_raw"], alpha=0.05, method="fdr_bh")
    merged.loc[valid_mask, "p_fdr"] = padj
else:
    merged.loc[valid_mask, "p_fdr"] = bh_fdr(merged.loc[valid_mask, "p_raw"].values)

merged["significant_raw"] = merged["p_raw"] < 0.05
merged["significant_fdr"] = merged["p_fdr"] < 0.05

# Direction labels — only label as stronger if significant
def direction(row):
    if not row["significant_fdr"]:
        return "similar"
    return "stronger_in_pediatric" if row["log_ratio"] > 0 else "stronger_in_adult"

merged["direction"] = merged.apply(direction, axis=1)

# Export full table
merged.to_csv(OUTPUT_DIR / "tables" / "table_shared_prr_ztest.csv", index=False)
print(f"Saved table_shared_prr_ztest.csv")
display(Markdown("**Sample of shared pairs (top 10 by |Z|):**"))
display(merged.nlargest(10, "Z", keep="all")[["drug","event","a_adult","a_ped","PRR_adult","PRR_ped","log_ratio","Z","p_fdr","direction"]])

In [ ]:
# Summary table
n_tested = valid_mask.sum()
n_sig_raw = int(merged["significant_raw"].sum())
n_sig_fdr = int(merged["significant_fdr"].sum())
n_stronger_ped = int((merged["direction"] == "stronger_in_pediatric").sum())
n_stronger_adult = int((merged["direction"] == "stronger_in_adult").sum())

valid_sub = merged[merged["logPRR_adult"].notna() & merged["logPRR_ped"].notna()
                   & np.isfinite(merged["logPRR_adult"]) & np.isfinite(merged["logPRR_ped"])]
pearson_r, pearson_p = pearsonr(valid_sub["logPRR_adult"], valid_sub["logPRR_ped"])
spearman_r, spearman_p = spearmanr(valid_sub["logPRR_adult"], valid_sub["logPRR_ped"])

summary_s2 = pd.DataFrame([
    {"Metric": "Shared pairs tested",                    "Value": f"{n_tested:,}"},
    {"Metric": "Significant (raw p<0.05)",                "Value": f"{n_sig_raw:,}  ({n_sig_raw/n_tested*100:.1f}%)"},
    {"Metric": "Significant (FDR<0.05)",                  "Value": f"{n_sig_fdr:,}  ({n_sig_fdr/n_tested*100:.1f}%)"},
    {"Metric": "Stronger in pediatric (FDR-sig)",         "Value": f"{n_stronger_ped:,}"},
    {"Metric": "Stronger in adult (FDR-sig)",             "Value": f"{n_stronger_adult:,}"},
    {"Metric": "Pearson r (log PRR adult vs ped)",        "Value": f"{pearson_r:.4f}  (p={pearson_p:.2e})"},
    {"Metric": "Spearman ρ",                              "Value": f"{spearman_r:.4f}  (p={spearman_p:.2e})"},
])
display(Markdown("**Section 2 Summary:**"))
display(summary_s2)
summary_s2.to_csv(OUTPUT_DIR / "tables" / "table_shared_prr_summary.csv", index=False)

In [ ]:
# Scatter plot: log(PRR_adult) vs log(PRR_ped)
x = valid_sub["logPRR_adult"].values
y = valid_sub["logPRR_ped"].values

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(x, y, alpha=0.15, s=6, color="#555", edgecolors="none")
lim = max(abs(x).max(), abs(y).max()) + 0.5
ax.plot([-lim, lim], [-lim, lim], "r--", alpha=0.6, label="y = x (equal strength)")
ax.set_xlabel("log(PRR) — Adult")
ax.set_ylabel("log(PRR) — Pediatric")
ax.set_title(f"Shared PRR Pairs (n={len(valid_sub):,})  Pearson r = {pearson_r:.3f}")
ax.legend(loc="upper left")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "S2_scatter_logPRR.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Volcano plot: log_ratio vs -log10(FDR)
vol = merged[merged["p_fdr"].notna() & np.isfinite(merged["p_fdr"]) & (merged["p_fdr"] > 0)].copy()
vol["neg_log10_fdr"] = -np.log10(vol["p_fdr"])

fig, ax = plt.subplots(figsize=(12, 8))

# Non-significant (grey)
ns = vol[~vol["significant_fdr"]]
ax.scatter(ns["log_ratio"], ns["neg_log10_fdr"], alpha=0.08, s=4, color="#BDBDBD", label="Not sig")

# Stronger in pediatric (orange, right)
sp = vol[(vol["direction"] == "stronger_in_pediatric")]
ax.scatter(sp["log_ratio"], sp["neg_log10_fdr"], alpha=0.25, s=8, color="#FF9800",
           label=f"Stronger in Pediatric ({len(sp):,})")

# Stronger in adult (blue, left)
sa = vol[(vol["direction"] == "stronger_in_adult")]
ax.scatter(sa["log_ratio"], sa["neg_log10_fdr"], alpha=0.25, s=8, color="#2196F3",
           label=f"Stronger in Adult ({len(sa):,})")

ax.axhline(-np.log10(0.05), color="red", ls="--", alpha=0.5, label="FDR = 0.05")
ax.axvline(0, color="black", ls="-", alpha=0.3)
ax.set_xlabel("log(PRR_pediatric / PRR_adult)   ← Adult stronger | Pediatric stronger →")
ax.set_ylabel("-log10(FDR p-value)")
ax.set_title("Volcano Plot — Shared PRR Pairs: Heterogeneity")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "S2_volcano.png", dpi=150, bbox_inches="tight")
plt.show()

### Section 2 — Interpretation

**Key thresholds for H1-2B support:**
- If a substantial proportion of shared pairs show **FDR < 0.05** with directional split → **supports H1-2B**
- If most pairs are "similar" (non-significant) → fails to support H1-2B
- Pearson / Spearman correlation: high correlation (>0.7) means similar ranking; low correlation means divergent

**Observation:** When a notable fraction of shared pairs shows significant heterogeneity by z-test
(FDR < 0.05) with asymmetric direction (e.g., many stronger in pediatric), this **supports H1-2B** —
same drug-AE pair has different signal strength between cohorts.

---
# Section 3 — Pooled vs Stratified PRR Signal Loss

Compare pooled PRR-positive set vs stratified (adult + pediatric) sets.
Tests **H1-2C**.

In [ ]:
# Compute missed signals
ped_missed = ped_set - pooled_set
adult_missed = adult_set - pooled_set
total_missed = ped_missed | adult_missed
pooled_only = pooled_set - adult_set - ped_set

ped_coverage = len(ped_set & pooled_set) / len(ped_set) if ped_set else 0
adult_coverage = len(adult_set & pooled_set) / len(adult_set) if adult_set else 0

table_s3 = pd.DataFrame([
    {"Category": "Adult PRR pairs",                         "Count": f"{len(adult_set):,}"},
    {"Category": "Pediatric PRR pairs",                     "Count": f"{len(ped_set):,}"},
    {"Category": "Pooled PRR pairs",                        "Count": f"{len(pooled_set):,}"},
    {"Category": "Pediatric signals missed by pooled",      "Count": f"{len(ped_missed):,}"},
    {"Category": "Adult signals missed by pooled",          "Count": f"{len(adult_missed):,}"},
    {"Category": "Total cohort-specific signals missed",    "Count": f"{len(total_missed):,}"},
    {"Category": "Pooled-only signals (not in either)",     "Count": f"{len(pooled_only):,}"},
    {"Category": "Pediatric coverage in pooled",            "Count": f"{ped_coverage*100:.1f}%"},
    {"Category": "Adult coverage in pooled",                "Count": f"{adult_coverage*100:.1f}%"},
])
display(Markdown("**Table PRR-Pooled-vs-Stratified:**"))
display(table_s3)
table_s3.to_csv(OUTPUT_DIR / "tables" / "table_prr_pooled_vs_stratified.csv", index=False)

# Bar plot — missed signals
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# Left: counts
cats1 = ["Adult\nstratified", "Pediatric\nstratified", "Pooled"]
vals1 = [len(adult_set), len(ped_set), len(pooled_set)]
axes[0].bar(cats1, vals1, color=["#2196F3", "#FF9800", "#9C27B0"], width=0.5, alpha=0.9)
for j, v in enumerate(vals1):
    axes[0].text(j, v + max(vals1)*0.01, f"{v:,}", ha="center", fontweight="bold")
axes[0].set_ylabel("PRR-positive pairs")
axes[0].set_title("Signal Counts: Stratified vs Pooled")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Right: missed signals
cats2 = ["Ped missed\nby pooled", "Adult missed\nby pooled", "Pooled-only\n(new)"]
vals2 = [len(ped_missed), len(adult_missed), len(pooled_only)]
axes[1].bar(cats2, vals2, color=["#FF9800", "#2196F3", "#F44336"], width=0.5, alpha=0.9)
for j, v in enumerate(vals2):
    axes[1].text(j, v + max(vals2)*0.01, f"{v:,}", ha="center", fontweight="bold")
axes[1].set_ylabel("Signals")
axes[1].set_title("Impact of Pooled Analysis")
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "S3_pooled_vs_stratified.png", dpi=150, bbox_inches="tight")
plt.show()

miss_pct = (len(ped_missed) + len(adult_missed)) / (len(adult_set) + len(ped_set)) * 100
print(f"\nTotal cohort-specific signals missed: {len(total_missed):,}")
print(f"  = {miss_pct:.1f}% of all stratified signals")

### Section 3 — Interpretation

**Key observations for H1-2C:**
- If **significant numbers** of pediatric or adult signals are missed by pooled analysis → **supports H1-2C**
- Especially concerning if pediatric coverage is much lower than adult coverage (due to Adult dataset size)
- `pooled-only` signals = new signals that appear ONLY in pooled analysis (possibly spurious from mixing)

**Observation:** Substantial missed signals (thousands) with differential pediatric/adult coverage
**supports H1-2C** — pooled PRR analysis loses cohort-specific patterns.

---
# Section 4 — Mini Conclusion: Hypothesis Support Summary

In [ ]:
# Derive verdicts programmatically
verdicts = {}

# H1-2A: overlap threshold
if jac < 0.30:
    verdicts["H1-2A"] = (True, f"SUPPORTED — Jaccard = {jac:.3f} < 0.30 (low overlap)")
elif jac < 0.60:
    verdicts["H1-2A"] = (True, f"PARTIAL SUPPORT — Jaccard = {jac:.3f} (moderate overlap)")
else:
    verdicts["H1-2A"] = (False, f"NOT SUPPORTED — Jaccard = {jac:.3f} ≥ 0.60 (high overlap)")

# H1-2B: FDR significance rate
fdr_rate = n_sig_fdr / n_tested if n_tested else 0
if fdr_rate >= 0.10:
    verdicts["H1-2B"] = (True, f"SUPPORTED — {n_sig_fdr:,} ({fdr_rate*100:.1f}%) shared pairs show FDR-significant heterogeneity")
elif fdr_rate >= 0.05:
    verdicts["H1-2B"] = (True, f"PARTIAL SUPPORT — {n_sig_fdr:,} ({fdr_rate*100:.1f}%) FDR-significant")
else:
    verdicts["H1-2B"] = (False, f"NOT SUPPORTED — only {n_sig_fdr:,} ({fdr_rate*100:.1f}%) FDR-significant")

# H1-2C: missed signal rate
missed_rate = len(total_missed) / (len(adult_set) + len(ped_set)) if (adult_set or ped_set) else 0
if len(total_missed) >= 1000 and missed_rate >= 0.05:
    verdicts["H1-2C"] = (True, f"SUPPORTED — {len(total_missed):,} ({missed_rate*100:.1f}%) cohort-specific signals missed by pooled")
elif missed_rate >= 0.01:
    verdicts["H1-2C"] = (True, f"PARTIAL SUPPORT — {len(total_missed):,} ({missed_rate*100:.1f}%) missed")
else:
    verdicts["H1-2C"] = (False, f"NOT SUPPORTED — only {len(total_missed):,} missed ({missed_rate*100:.1f}%)")

verdict_table = pd.DataFrame([
    {"Hypothesis": "H1-2A (Signal set overlap)", "Verdict": verdicts["H1-2A"][1]},
    {"Hypothesis": "H1-2B (Shared signal strength)", "Verdict": verdicts["H1-2B"][1]},
    {"Hypothesis": "H1-2C (Pooled vs stratified)", "Verdict": verdicts["H1-2C"][1]},
])
display(Markdown("### Hypothesis Verdict Table"))
display(verdict_table)
verdict_table.to_csv(OUTPUT_DIR / "tables" / "hypothesis_verdicts.csv", index=False)

# Overall conclusion
all_supported = all(v[0] for v in verdicts.values())
most_supported = sum(v[0] for v in verdicts.values()) >= 2

print("\n" + "=" * 70)
print("OVERALL VERDICT")
print("=" * 70)
if all_supported:
    print("ALL 3 sub-hypotheses SUPPORTED.")
    print("→ Strong evidence to REJECT H0-2")
    print('→ Conclusion: "Adult and Pediatric data SHOULD NOT be pooled" is SUPPORTED')
elif most_supported:
    print("Majority (2 of 3) sub-hypotheses supported.")
    print("→ Moderate evidence against pooling")
else:
    print("Insufficient support across sub-hypotheses.")
    print("→ Fail to reject H0-2")
print("=" * 70)

## Final Interpretation

### Hypothesis Summary

| Sub-Hypothesis | What it tests | Supporting evidence |
|----------------|---------------|---------------------|
| **H1-2A** | Signal set overlap between cohorts | Jaccard, Overlap coefficient |
| **H1-2B** | Shared signal magnitude heterogeneity | z-test + FDR on log(PRR) |
| **H1-2C** | Pooled vs stratified signal loss | Missed signal counts |

### Conclusion (derived from data above)

The analysis results above programmatically evaluate each sub-hypothesis.
If **all three** are supported, the overall conclusion is:

> **Pediatric and adult FAERS data should NOT be pooled.**  
> Cohort-stratified analysis preserves age-specific safety signal patterns that
> pooled analysis would obscure.

### Limitations
- FAERS is a spontaneous reporting database — disproportionality ≠ causality
- Z-test assumes asymptotic normality of log(PRR) — may be imprecise for low-count pairs
- BH-FDR controls expected false discovery proportion, not family-wise error
- Missing pairs in pooled may reflect power (sample size) rather than true biological difference